PREPROCESSING


In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse
from create_engine import ENGINE

query = """
SELECT *
FROM IDS_DATA
"""

df = pd.read_sql(query, ENGINE)

print(df.tail())

2026-05-26 13:47:01,931 - INFO - Connection Built Successfully


Connecting to DB: localhost/cti_fyp
                                        flow id      src ip  src port  \
1066692  172.16.0.1-192.168.10.50-62393-20000-6  172.16.0.1     62393   
1066693     172.16.0.1-192.168.10.50-35529-80-6  172.16.0.1     35529   
1066694   172.16.0.1-192.168.10.50-64318-2222-6  172.16.0.1     64318   
1066695   172.16.0.1-192.168.10.50-33248-9040-6  172.16.0.1     33248   
1066696   172.16.0.1-192.168.10.50-44033-1272-6  172.16.0.1     44033   

                dst ip  dst port  protocol               timestamp  \
1066692  192.168.10.50     20000         6  07/07/2017 07:52:08 PM   
1066693  192.168.10.50        80         6  07/07/2017 09:11:28 PM   
1066694  192.168.10.50      2222         6  07/07/2017 08:09:11 PM   
1066695  192.168.10.50      9040         6  07/07/2017 07:54:38 PM   
1066696  192.168.10.50      1272         6  07/07/2017 07:52:03 PM   

         flow duration  total fwd packet  total bwd packets  ...  \
1066692             52              

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df.isnull().sum().sum()

np.int64(0)

In [4]:
df.duplicated().sum()

np.int64(2)

In [5]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [6]:
df.dropna(inplace=True)

In [8]:
df.drop_duplicates(keep='first', inplace=True)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1066344 entries, 0 to 1066343
Data columns (total 84 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   flow id                     1066344 non-null  object 
 1   src ip                      1066344 non-null  object 
 2   src port                    1066344 non-null  int64  
 3   dst ip                      1066344 non-null  object 
 4   dst port                    1066344 non-null  int64  
 5   protocol                    1066344 non-null  int64  
 6   timestamp                   1066344 non-null  object 
 7   flow duration               1066344 non-null  int64  
 8   total fwd packet            1066344 non-null  int64  
 9   total bwd packets           1066344 non-null  int64  
 10  total length of fwd packet  1066344 non-null  float64
 11  total length of bwd packet  1066344 non-null  float64
 12  fwd packet length max       1066344 non-null  float64
 1

In [69]:
df_new.shape

(1031402, 78)

In [10]:
df=df.reset_index(drop=True)

In [13]:
drop_cols=["flow id", "src ip","dst ip", "src port", "dst port", "timestamp"]

In [14]:
df.columns

Index(['flow id', 'src ip', 'src port', 'dst ip', 'dst port', 'protocol',
       'timestamp', 'flow duration', 'total fwd packet', 'total bwd packets',
       'total length of fwd packet', 'total length of bwd packet',
       'fwd packet length max', 'fwd packet length min',
       'fwd packet length mean', 'fwd packet length std',
       'bwd packet length max', 'bwd packet length min',
       'bwd packet length mean', 'bwd packet length std', 'flow bytes/s',
       'flow packets/s', 'flow iat mean', 'flow iat std', 'flow iat max',
       'flow iat min', 'fwd iat total', 'fwd iat mean', 'fwd iat std',
       'fwd iat max', 'fwd iat min', 'bwd iat total', 'bwd iat mean',
       'bwd iat std', 'bwd iat max', 'bwd iat min', 'fwd psh flags',
       'bwd psh flags', 'fwd urg flags', 'bwd urg flags', 'fwd header length',
       'bwd header length', 'fwd packets/s', 'bwd packets/s',
       'packet length min', 'packet length max', 'packet length mean',
       'packet length std', 'packet len

In [15]:
# drop irrelevant cols
df_new = df.drop(drop_cols, axis=1)

# remove benign duplicates only
feature_cols = [c for c in df_new.columns if c != 'label']

benign = (
    df_new[df_new['label'] == 'BENIGN']
    .drop_duplicates(subset=feature_cols)
)

attack = df_new[df_new['label'] != 'BENIGN']

# combine
df_new = pd.concat([benign, attack], axis=0)

# reset index
df_new.reset_index(drop=True, inplace=True)

In [17]:
df_new.columns

Index(['protocol', 'flow duration', 'total fwd packet', 'total bwd packets',
       'total length of fwd packet', 'total length of bwd packet',
       'fwd packet length max', 'fwd packet length min',
       'fwd packet length mean', 'fwd packet length std',
       'bwd packet length max', 'bwd packet length min',
       'bwd packet length mean', 'bwd packet length std', 'flow bytes/s',
       'flow packets/s', 'flow iat mean', 'flow iat std', 'flow iat max',
       'flow iat min', 'fwd iat total', 'fwd iat mean', 'fwd iat std',
       'fwd iat max', 'fwd iat min', 'bwd iat total', 'bwd iat mean',
       'bwd iat std', 'bwd iat max', 'bwd iat min', 'fwd psh flags',
       'bwd psh flags', 'fwd urg flags', 'bwd urg flags', 'fwd header length',
       'bwd header length', 'fwd packets/s', 'bwd packets/s',
       'packet length min', 'packet length max', 'packet length mean',
       'packet length std', 'packet length variance', 'fin flag count',
       'syn flag count', 'rst flag count',

In [18]:
df_new.duplicated().sum()

np.int64(158545)

In [19]:
# df_new.select_dtypes(include="object")
df_new["label"] = df_new["label"].str.replace('- Attempted', '', regex=False).str.strip()

In [20]:
df_new["label"].unique()

array(['BENIGN', 'FTP-Patator', 'SSH-Patator', 'DoS slowloris',
       'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye', 'Heartbleed',
       'Bot', 'PortScan', 'DDoS'], dtype=object)

In [21]:
from sklearn.preprocessing import LabelEncoder

In [22]:
le = LabelEncoder()
df_new['label'] = le.fit_transform(df_new['label'])

In [23]:
df_new["label"].value_counts()

label
0     590555
4     159048
9     159023
2      95123
3       7647
6       5707
5       5109
7       3984
10      2988
1       2207
8         11
Name: count, dtype: int64

In [24]:
df_new.isnull().sum().sum()

np.int64(0)

In [27]:
df_new['label'].tail()

1031397    9
1031398    2
1031399    9
1031400    9
1031401    9
Name: label, dtype: int64

In [36]:
df_new.describe()

,protocol,flow duration,total fwd packet,total bwd packets,total length of fwd packet,total length of bwd packet,fwd packet length max,fwd packet length min,fwd packet length mean,fwd packet length std,...,fwd seg size min,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min,label
count,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,...,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06,1.031402e+06
mean,9.905708e+00,1.092931e+07,1.110424e+01,1.220112e+01,5.015277e+02,1.881260e+04,2.011501e+02,1.598661e+01,4.454367e+01,6.153750e+01,...,1.957891e+01,1.451487e+05,5.775595e+04,2.354974e+05,1.110613e+05,4.069521e+06,3.075474e+05,4.351888e+06,3.783796e+06,2.327352e+00
std,5.269525e+00,2.872269e+07,7.509408e+02,1.002310e+03,4.577113e+03,2.240301e+06,4.726521e+02,3.286057e+01,9.257427e+01,1.513324e+02,...,1.054594e+01,8.155619e+05,4.778719e+05,1.209346e+06,7.346736e+05,1.313868e+07,2.848895e+06,1.388994e+07,1.286681e+07,3.283274e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.000000e+00,2.790000e+02,1.000000e+00,1.000000e+00,2.000000e+01,9.500000e+01,2.000000e+01,0.000000e+00,2.500000e+00,0.000000e+00,...,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,6.000000e+00,9.114700e+04,4.000000e+00,2.000000e+00,7.600000e+01,2.640000e+02,4.400000e+01,0.000000e+00,3.900000e+01,0.000000e+00,...,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.700000e+01,4.523040e+06,8.000000e+00,6.000000e+00,3.550000e+02,1.159500e+04,3.280000e+02,3.600000e+01,5.000000e+01,1.119585e+02,...,2.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e+00
max,1.700000e+01,1.200000e+08,2.079630e+05,2.846030e+05,2.429858e+06,6.270000e+08,2.482000e+04,1.472000e+03,5.775500e+03,7.018511e+03,...,4.400000e+01,1.100975e+08,7.420000e+07,1.100975e+08,1.100975e+08,1.200000e+08,7.690000e+07,1.200000e+08,1.200000e+08,1.000000e+01


In [47]:
iat_columns = [
        'flow iat min',
        'flow iat max', 
        'flow iat mean',
        'fwd iat min',
        'bwd iat min',
        'flow duration',
        'flow packets/s' 
    ]
    
for col in iat_columns:
    if col in df_new.columns:
            # Count negatives
        neg_count = (df_new[col] < 0).sum()
            
        if neg_count > 0:
                print(f"⚠️ Found {neg_count} negative values in {col}")
                
                # Check severity
                severe = (df_new[col] < -1).sum()
                
                if severe > 0:
                    print(f"  🔴 {severe} values < -1 (serious corruption)")
                
                # Fix: Replace negative with 0
                    df_new.loc[df_new[col] < 0, col] = 0
                    print(f"  ✅ Replaced with 0")

In [44]:
num_cols = df_new.select_dtypes(include=[np.number]).columns
neg_cols = (df_new[num_cols] > 0).sum()

In [46]:
neg_cols

protocol                      1030799
flow duration                 1031397
total fwd packet              1031389
total bwd packets             1022633
total length of fwd packet     838417
                               ...   
idle mean                      225849
idle std                        92246
idle max                       225849
idle min                       225849
label                          440847
Length: 78, dtype: int64

In [67]:
df_new.columns

Index(['protocol', 'flow duration', 'total fwd packet', 'total bwd packets',
       'total length of fwd packet', 'total length of bwd packet',
       'fwd packet length max', 'fwd packet length min',
       'fwd packet length mean', 'fwd packet length std',
       'bwd packet length max', 'bwd packet length min',
       'bwd packet length mean', 'bwd packet length std', 'flow bytes/s',
       'flow packets/s', 'flow iat mean', 'flow iat std', 'flow iat max',
       'flow iat min', 'fwd iat total', 'fwd iat mean', 'fwd iat std',
       'fwd iat max', 'fwd iat min', 'bwd iat total', 'bwd iat mean',
       'bwd iat std', 'bwd iat max', 'bwd iat min', 'fwd psh flags',
       'bwd psh flags', 'fwd urg flags', 'bwd urg flags', 'fwd header length',
       'bwd header length', 'fwd packets/s', 'bwd packets/s',
       'packet length min', 'packet length max', 'packet length mean',
       'packet length std', 'packet length variance', 'fin flag count',
       'syn flag count', 'rst flag count',

In [55]:
print((df_new == -np.inf).sum().sum())

0


In [62]:
import logging

logging.basicConfig(
    level=logging.INFO,  
    format="%(asctime)s - %(levelname)s - %(message)s"
)

df_new.to_sql(
    name="IDS_DATA_FINALIZED",
    con=ENGINE,
    if_exists="fail",
    index=False,
    chunksize=10000
)
logging.info(f"Data stored in mysql successfully............")

C:\Users\UMAR.TECH\AppData\Local\Temp\ipykernel_28848\3652951596.py:8: UserWarning: The provided table name 'IDS_DATA_FINALIZED' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_new.to_sql(
2026-05-26 14:42:39,092 - INFO - Data stored in mysql successfully............


In [ ]:
df = pd.read_sql("SELECT * FROM IDS_DATA_FINALIZED", ENGINE)
print(df.head())

   protocol  flow duration  total fwd packet  total bwd packets  \
0         6       57185951               164                205   
1        17          30996                 1                  1   
2        17         130551                 1                  1   
3        17            173                 2                  2   
4         6      115455124                17                 16   

   total length of fwd packet  total length of bwd packet  \
0                     11307.0                    273722.0   
1                        57.0                        73.0   
2                        48.0                        64.0   
3                        90.0                       122.0   
4                      2129.0                       295.0   

   fwd packet length max  fwd packet length min  fwd packet length mean  \
0                 1159.0                    0.0               68.945122   
1                   57.0                   57.0               57.000000   
2    